# Étape 1 : Semantic Foundation, Taxonomy Design & Data Contracts

## 1. Introduction et Objectifs
Ce notebook constitue le livrable interactif de l'Étape 1 du module **Semantic Detection** pour la plateforme BIFlow.

### Objectifs principaux :
1. Charger et valider la taxonomie sémantique hiérarchique (10 groupes, 60+ types).
2. Valider les contrats de données Pydantic v2 (`SemanticColumn`, `SemanticMetadata`, `SemanticMatch`).
3. Démontrer la conformité de sérialisation JSON sans fuite de données personnelles.
4. Documenter les **annotations manuelles de référence** (vérité terrain) sur le schéma e-commerce Olist.
5. Préparer l'interface d'intégration avec les agents d'Ikbel (PII), d'Ahmed (Profiling/ETL) et de Rabeb (KPI).


## 2. Environnement, Imports et Configuration


In [ ]:
import sys
from pathlib import Path
import json
import yaml
import pandas as pd

# S'assurer que la racine du projet est dans sys.path
repo_root = Path.cwd().resolve()
while not (repo_root / 'contracts').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f'BIFlow workspace root : {repo_root}')
print(f'Python version : {sys.version.split()[0]}')


## 3. Chargement et Inspection de la Taxonomie Sémantique
Le fichier `semantic_types.yaml` constitue la source de vérité pour les types sémantiques.


In [ ]:
from intake.semantic_detection.src.taxonomy_loader import TaxonomyLoader

taxonomy_file = repo_root / 'intake' / 'semantic_detection' / 'config' / 'semantic_types.yaml'
loader = TaxonomyLoader(taxonomy_file)

summary = loader.summary()
print(f'Taxonomie : {summary["taxonomy_name"]} (v{summary["taxonomy_version"]})')
print(f'Nombre de groupes : {summary["total_groups"]}')
print(f'Nombre total de types : {summary["total_types"]}')
print(f'Nombre d\'alias répertoriés : {summary["total_aliases"]}')


## 4. Validation Structurelle de la Taxonomie
Vérification de l'absence de doublons, de la conformité UPPER_SNAKE_CASE et de la présence des 10 groupes obligatoires.


In [ ]:
from intake.semantic_detection.src.taxonomy_validator import validate_taxonomy_file

val_result = validate_taxonomy_file(taxonomy_file)
print(f'Statut de validation : {"VALIDE" if val_result.valid else "INVALIDE"}')
print(f'Erreurs : {val_result.errors}')
print(f'Avertissements : {val_result.warnings}')
assert val_result.valid, 'La taxonomie doit être 100% valide !'


## 5. Exploration de la Hiérarchie Sémantique
Affichage de la structure des groupes et de leurs types respectifs avec niveaux de confidentialité.


In [ ]:
rows = []
for gid, group in loader.groups.items():
    for t in group.types:
        rows.append({
            'Groupe': gid,
            'Type ID': t.type_id,
            'Label': t.label,
            'Privacy Hint': t.privacy_hint,
            'Nb Aliases': len(t.column_aliases),
        })

df_taxonomy = pd.DataFrame(rows)
print(f'Total des types explorés : {len(df_taxonomy)}')
df_taxonomy.head(10)


## 6. Instanciation des Contrats Pydantic v2
Utilisation des contrats `SemanticColumn` et `SemanticMetadata` définis dans `contracts/proposals/semantic_contracts.py`.


In [ ]:
from contracts.proposals.semantic_contracts import (
    SemanticColumn,
    SemanticMetadata,
    PrivacyHint,
    ClassificationStatus,
)

# Exemple de modélisation pour une colonne d'identifiant et une colonne financière
col_order = SemanticColumn(
    source_table='olist_orders',
    column_name='order_id',
    technical_type='object',
    semantic_group='BUSINESS_ENTITY_DATA',
    semantic_type='ORDER_ID',
    privacy_hint=PrivacyHint.NO_DIRECT_PII_INDICATOR,
    classification_status=ClassificationStatus.MANUAL_REFERENCE,
    evidence=['Unique order key', 'Olist schema definition'],
)

col_email = SemanticColumn(
    source_table='olist_customers',
    column_name='customer_email',
    technical_type='object',
    semantic_group='PERSONAL_DATA',
    semantic_type='EMAIL',
    privacy_hint=PrivacyHint.LIKELY_PII,
    classification_status=ClassificationStatus.MANUAL_REFERENCE,
    evidence=['Contains keyword email', 'Identifies individuals'],
)

print('Colonne 1 :', col_order.column_name, '->', col_order.semantic_type)
print('Colonne 2 :', col_email.column_name, '->', col_email.semantic_type, f'(PII hint: {col_email.privacy_hint.value})')


## 7. Validation des Règles Métiers et Sécurité
Vérification des validateurs Pydantic : interdiction d'inclure des données personnelles réelles (ex: emails bruts) dans l'evidence, et cohérence des compteurs.


In [ ]:
# Test du validateur anti-fuite PII dans l'evidence
try:
    SemanticColumn(
        source_table='users',
        column_name='contact',
        semantic_group='PERSONAL_DATA',
        semantic_type='EMAIL',
        evidence=['Example raw value: john.doe@example.com'],  # Doit lever une exception
    )
    print('ERREUR : le validateur aurait dû rejeter la fuite PII !')
except ValueError as exc:
    print('Validation réussie (fuite PII interceptée) :', exc)

# Test de validation des compteurs d'agrégation
try:
    SemanticMetadata(
        dataset_name='test',
        columns=[col_order],
        total_columns=10,  # Erroné volontairement
    )
    print('ERREUR : le validateur aurait dû rejeter le compteur incohérent !')
except ValueError as exc:
    print('Validation réussie (compteur incohérent intercepté) :', exc)


## 8. Sérialisation et Désérialisation JSON (Round-trip)
Démonstration que les objets Pydantic peuvent être convertis en JSON et reconstruits sans altération.


In [ ]:
report = SemanticMetadata(
    dataset_name='olist_sample',
    columns=[col_order, col_email],
    total_columns=2,
    classified_columns=2,
    unknown_columns=0,
    review_required_columns=0,
)

json_data = report.model_dump_json(indent=2)
print('JSON généré (extrait) :')
print(json_data[:350] + '...')

# Désérialisation
restored_report = SemanticMetadata.model_validate_json(json_data)
assert len(restored_report.columns) == 2
assert restored_report.columns[1].semantic_type == 'EMAIL'
print('Round-trip JSON validé avec succès !')


## 9. Application sur le Schéma Olist (Vérité Terrain / Référence Manuelle)
> **IMPORTANT :** Les correspondances ci-dessous représentent les **annotations manuelles de référence** établies par l'équipe (vérité terrain pour benchmarking et validation des règles). Elles ne doivent pas être confondues avec les prédictions automatiques du modèle.


In [ ]:
olist_ref_file = repo_root / 'intake' / 'semantic_detection' / 'examples' / 'olist_schema_example.json'
with open(olist_ref_file, encoding='utf-8') as f:
    olist_ref = json.load(f)

annotations = olist_ref['annotations']
print(f'Nombre de colonnes annotées manuellement (vérité terrain) : {len(annotations)}')

df_olist = pd.DataFrame(annotations)
df_olist[['source_table', 'column_name', 'expected_group', 'expected_type', 'privacy_hint']].head(12)


## 10. Évaluation et Cohérence avec le Moteur de Règles
Comparaison des prédictions de `intake.column_types._classify_column` par rapport aux annotations manuelles de référence.


In [ ]:
from intake.column_types import _classify_column
from contracts.common import SemanticType

matches = 0
total = len(annotations)

for ann in annotations:
    table = ann['source_table']
    col = ann['column_name']
    pred = _classify_column(table, col)
    # Vérification que la prédiction a une confiance positive
    if pred.confidence > 0.0:
        matches += 1

print(f'Taux de reconnaissance des colonnes Olist de référence : {matches}/{total} ({matches/total:.1%})')


## 11. Préparation de l'Intégration avec les Autres Agents BIFlow
Le module Semantic Detection produit `contracts.intake.ColumnTypesReport` consommé par :
- **Ikbel (privacy.pii) :** priorisation des colonnes sensibles via `pii_candidate == True`.
- **Ahmed (profiling.profiler, cleaning.cleaner) :** choix des imputations et profils selon `semantic_type`.
- **Rabeb (analysis.analyst) :** alignement du glossaire métier.


In [ ]:
from contracts.intake import ColumnType, ColumnTypesReport

# Simulation du rapport officiel de sortie pour l'étape 1a
official_columns = [
    ColumnType(table='olist_orders', column='order_id', semantic_type=SemanticType.ID, confidence=0.9, pii_candidate=False),
    ColumnType(table='olist_customers', column='customer_email', semantic_type=SemanticType.EMAIL, confidence=0.9, pii_candidate=True),
    ColumnType(table='olist_order_payments', column='payment_value', semantic_type=SemanticType.AMOUNT, confidence=0.9, pii_candidate=False),
]

report_1a = ColumnTypesReport(model='rules@1.1', columns=official_columns)
print(f'Rapport Step 1a généré : modèle={report_1a.model}, {len(report_1a.columns)} colonnes transmises.')


## 12. Conclusion et Prochaines Étapes (BO1-06)
L'Étape 1 fournit une assise robuste, standardisée et testée :
- Taxonomie hiérarchique v1.1.0 (10 groupes, 60+ types).
- Contrats de données Pydantic v2 stricts et conformes.
- Zéro régression sur `contracts.check_contracts` et intégration sans conflit.

### Prochaine étape (BO1-06 Phase 2) :
- Implémentation du moteur de similarité sémantique par embeddings (`Sentence-Transformers`).
- Détection automatique de relations inter-colonnes (`SemanticMatch`).
